In [1]:
import ee
import geemap
ee.Authenticate()  # Run once, it will prompt you to authorize with your Google account
ee.Initialize()


Enter verification code: 4/1AVGzR1DBWztmKn3T2-YBMd4_MpprFDXQMWErjjedP0f5fL-UMGygOLHFkdY

Successfully saved authorization token.


In [2]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point


# Load Excel and convert
df = pd.read_excel("branner i innmark og utmark 2016-2024.xlsx")
df['Dato anrop'] = pd.to_datetime(df['Dato anrop'])
df['Total Area (decare)'] = df['Brannskadd areal produktiv skog (dekar/mål)'].fillna(0) + \
df['Brannskadd areal uproduktiv skog og annen utmark (dekar/mål)'].fillna(0)


# Sort by largest burned area 
df = df.sort_values('Total Area (decare)', ascending=False)


# Convert to GeoDataFrame
fire_gdf = gpd.GeoDataFrame(df,
geometry=gpd.points_from_xy(df['Geolokasjon lengdegrad'], df['Geolokasjon breddegrad']),
crs="EPSG:4326")

In [3]:
fire_gdf

,Oppdragstype,Kommunenavn,Fylke,Geolokasjon breddegrad,Geolokasjon lengdegrad,Ansvarlig brannvesen i hendelseskommune,110-sentral,Time på døgnet,Dato anrop,Responstid,...,Har brann- og redningsvesenet aktivt reddet personer ved innsatsen?,Førte hendelsen eller håndteringen av denne til forstyrrelser i kritisk infrastruktur?,Omkom dyr under hendelsen?,Brannskadd areal produktiv skog (dekar/mål),Brannskadd areal uproduktiv skog og annen utmark (dekar/mål),Var det gassflasker eller gasstanker i eller i området rundt brannobjektet som utgjorde en potensiell fare?,Opplevde brann- og redningsvesenet angrep eller trusler i løpet av innsatsen?,Er hendelsen eller håndteringen av en slik karakter at det bør legges ressurser i å videreformidle erfaringer til andre brann- og redningsvesen?,Total Area (decare),geometry
9485,Brann i skog- eller utmark,Indre Østfold,Østfold,59.501772,11.498272,Indre Østfold brann og redning IKS,Øst 110-sentral IKS,16,2023-05-30,00:59:22,...,Nei,Nei,Nei,15000.0,5000.0,Nei,Nei,Nei,20000.0,POINT (11.49827 59.50177)
9655,Brann i skog- eller utmark,Larvik,Vestfold,59.211349,9.868284,Larvik brannvesen,Sør-Øst 110 IKS,16,2023-06-15,NaN,...,Nei,Nei,Vet ikke,10000.0,10000.0,Nei,Nei,Ja,20000.0,POINT (9.86828 59.21135)
3243,Brann i gress- eller innmark,Time,Rogaland,58.742526,5.746153,Rogaland brann og redning IKS,110 Sør-Vest,16,2019-04-15,00:15:18,...,Nei,NaN,Nei,10000.0,10000.0,Nei,Nei,Nei,20000.0,POINT (5.74615 58.74253)
9368,Brann i gress- eller innmark,Time,Rogaland,58.743592,5.743875,Rogaland brann og redning IKS,110 Sør-Vest,0,2023-04-07,00:17:28,...,Nei,Nei,Nei,20.0,15000.0,Nei,Nei,Nei,15020.0,POINT (5.74387 58.74359)
9590,Brann i gress- eller innmark,Alta,Finnmark,69.964998,23.439698,Alta brann og redning,110-sentralen for Finnmark,17,2023-06-16,00:27:05,...,Nei,Nei,Vet ikke,10000.0,1.0,Vet ikke,Nei,Nei,10001.0,POINT (23.4397 69.965)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4590,Brann i gress- eller innmark,Austrheim,Vestland,60.788506,4.938952,Nordhordland brann og redning,110 Vest,17,2019-05-13,00:13:46,...,Nei,NaN,Nei,0.0,0.0,Nei,Nei,Nei,0.0,POINT (4.93895 60.78851)
4588,Brann i gress- eller innmark,Bodø,Nordland,67.274675,14.377815,Salten Brann IKS,110 Nordland,16,2019-04-29,00:10:29,...,Nei,NaN,Nei,0.0,0.0,Nei,Nei,Nei,0.0,POINT (14.37782 67.27468)
4586,Brann i gress- eller innmark,Bergen,Vestland,60.419294,5.469734,Bergen brannvesen,110 Vest,18,2019-04-23,00:07:17,...,Nei,NaN,Nei,0.0,0.0,Nei,Nei,Nei,0.0,POINT (5.46973 60.41929)
4585,Brann i gress- eller innmark,Nordre Follo,Akershus,59.799487,10.776087,Follo brannvesen IKS,Øst 110-sentral IKS,17,2019-04-23,00:33:50,...,Nei,NaN,Nei,0.0,0.0,Nei,Nei,Nei,0.0,POINT (10.77609 59.79949)


In [ ]:
""""
from datetime import timedelta
import os


def export_sentinel_images(event, index, export_folder='SENTINEL_EXPORTS'):
    try:
        point = ee.Geometry.Point([event.geometry.x, event.geometry.y])
        region = point.buffer(1000).bounds()
        event_date = event['Dato anrop']
        start = ee.Date(event_date - timedelta(days=10))
        end = ee.Date(event_date)


        # Sentinel-1 VV
        s1 = ee.ImageCollection("COPERNICUS/S1_GRD") \
        .filterBounds(region) \
        .filterDate(start, end) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .median().select('VV')


        # Sentinel-2 RGB
        s2 = ee.ImageCollection("COPERNICUS/S2") \
        .filterBounds(region) \
        .filterDate(start, end) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
        .median().select(['B4', 'B3', 'B2'])


        # Export to GeoTIFF
        os.makedirs(export_folder, exist_ok=True)
        geemap.ee_export_image(s1, filename=os.path.join(export_folder, f's1_fire_{index:03d}.tif'),
        scale=10, region=region, file_per_band=False)
        geemap.ee_export_image(s2, filename=os.path.join(export_folder, f's2_fire_{index:03d}.tif'),
        scale=10, region=region, file_per_band=False)


        print(f"✅ Exported Sentinel data for fire {index+1}/{len(fire_gdf)}")


    except Exception as e:
        print(f"❌ Error exporting fire {index+1}: {e}")


for i, row in fire_gdf.iterrows():
    export_sentinel_images(row, i)

In [4]:
from datetime import timedelta  
def extract_veg_temp_indices(event):
    point = ee.Geometry.Point([event.geometry.x, event.geometry.y])
    
    region = point.buffer(1000).bounds()
    event_date = event['Dato anrop']
    start = ee.Date(event_date - timedelta(days=20))
    end = ee.Date(event_date)


    s2 = ee.ImageCollection("COPERNICUS/S2") \
    .filterBounds(region) \
    .filterDate(start, end) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))

    if s2.size().getInfo() == 0:
        raise ValueError("No Sentinel-2 images available for this region and date.")

    s2 = s2.median()


    ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = s2.normalizedDifference(['B3', 'B8']).rename('NDWI')
    msi = s2.normalizedDifference(['B11', 'B8']).rename('MSI')


    temp = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY") \
    .filterDate(start, end).select('temperature_2m').mean()


    image = ee.Image.cat([ndvi, ndwi, msi, temp])
    values = image.reduceRegion(reducer=ee.Reducer.mean(), geometry=region, scale=1000).getInfo()


    return {
        'NDVI': values.get('NDVI'),
        'NDWI': values.get('NDWI'),
        'MSI': values.get('MSI'),
        'Temperature': values.get('temperature_2m')
    }

In [5]:
def extract_terrain_landcover(event):
    point = ee.Geometry.Point([event.geometry.x, event.geometry.y])
    region = point.buffer(1000).bounds()


    dem = ee.Image("USGS/SRTMGL1_003")
    slope = ee.Terrain.slope(dem)
    aspect = ee.Terrain.aspect(dem)
    elevation = dem


    landcover = ee.Image("ESA/WorldCover/v100/2020").select('Map')
    tree_cover = ee.Image("UMD/hansen/global_forest_change_2022_v1_10").select('treecover2000')


    image = ee.Image.cat([elevation, slope, aspect, landcover, tree_cover])
    values = image.reduceRegion(reducer=ee.Reducer.mean(), geometry=region, scale=30).getInfo()


    return {
        'Elevation': values.get('elevation'),
        'Slope': values.get('slope'),
        'Aspect': values.get('aspect'),
        'LandCover': values.get('Map'),
        'TreeCover': values.get('treecover2000')
    }

In [6]:
def extract_weather(event):
    point = ee.Geometry.Point([event.geometry.x, event.geometry.y])
    region = point.buffer(1000).bounds()
    date = ee.Date(event['Dato anrop'])


    daily = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY") \
    .filterDate(date, date.advance(1, 'day'))
    last3 = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY") \
    .filterDate(date.advance(-3, 'day'), date)


    humidity = daily.select('dewpoint_temperature_2m').mean()
    wind = daily.select('u_component_of_wind_10m').mean()
    precip = last3.select('total_precipitation').sum()


    image = ee.Image.cat([humidity, wind, precip])
    values = image.reduceRegion(reducer=ee.Reducer.mean(), geometry=region, scale=1000).getInfo()


    return {
        'Humidity': values.get('dewpoint_temperature_2m'),
        'WindSpeed': values.get('u_component_of_wind_10m'),
        'Precip3Days': values.get('total_precipitation')
    }

In [7]:
def extract_population(event):
    point = ee.Geometry.Point([event.geometry.x, event.geometry.y])
    region = point.buffer(1000).bounds()


    pop = ee.ImageCollection("WorldPop/GP/100m/pop") \
    .filterDate('2020-01-01', '2020-12-31').median()
    val = pop.reduceRegion(reducer=ee.Reducer.mean(), geometry=region, scale=100).getInfo()


    return {
        'Population': val.get('population')
    }

In [8]:
def safe_min_distance(gdf, pt):
    """Compute min distance safely, return None if empty or NaN."""
    if gdf is None or gdf.empty:
        return None
    dists = gdf.distance(pt)
    if dists.isnull().all():
        return None
    val = dists.min()
    return None if pd.isna(val) else float(val)


In [9]:
"""
import osmnx as ox

def get_distance_to_roads(lat, lon, radius_m=5000):
    try:
        # Query roads within radius (meters)
        roads = ox.features_from_point((lat, lon), tags={'highway': True}, dist=radius_m)
        if roads.empty:
            return None

        # Compute min great-circle distance in meters
        min_dist = roads.geometry.apply(
            lambda g: ox.distance.great_circle_vec(lat, lon, g.centroid.y, g.centroid.x)
        ).min()
        return float(min_dist)

    except Exception as e:
        print(f"❌ Road distance error at {lat},{lon}: {e}")
        return None


def get_distance_to_settlements(lat, lon, radius_m=5000):
    try:
        settlements = ox.features_from_point((lat, lon), tags={'building': True}, dist=radius_m)
        if settlements.empty:
            return None

        min_dist = settlements.geometry.apply(
            lambda g: ox.distance.great_circle_vec(lat, lon, g.centroid.y, g.centroid.x)
        ).min()
        return float(min_dist)

    except Exception as e:
        print(f"❌ Settlement distance error at {lat},{lon}: {e}")
        return None


SyntaxError: incomplete input (3401763985.py, line 1)

In [10]:
import osmnx as ox

lat, lon = fire_gdf.iloc[0].geometry.y, fire_gdf.iloc[0].geometry.x

try:
    # Query roads within 5000 meters of the fire point
    roads = ox.features_from_point((lat, lon), tags={'highway': True}, dist=5000)
    print("✅ Roads retrieved:", len(roads))
    print("Columns:", roads.columns.tolist())
except Exception as e:
    print("❌ OSM query failed:", e)


✅ Roads retrieved: 843
Columns: ['geometry', 'highway', 'name', 'ref:nsrq', 'wheelchair', 'capacity:hgv', 'capacity:motorcar', 'description:no', 'hgv', 'agricultural', 'bicycle', 'foot', 'lit', 'maxspeed', 'motorroad', 'ref', 'surface', 'trailblazed', 'motor_vehicle', 'piste:type', 'class:bicycle:mtb', 'source', 'mtb:scale', 'trail_visibility', 'oneway', 'lanes', 'maxheight', 'junction', 'bridge', 'bridge:name', 'layer', 'maxweight:signed', 'shoulder', 'bridge:description', 'sidewalk', 'toll', 'tracktype', 'fixme', 'man_made', 'access', 'service', 'route', 'lanes:backward', 'lanes:forward', 'turn:lanes:backward', 'turn:lanes:forward', 'piste:grooming', 'snowplowing', 'cutline', 'area', 'note:no', 'note', 'waterway', 'horse', 'tunnel', 'smoothness', 'covered', 'handrail', 'tactile_paving', 'flood_prone', 'incline']


In [11]:
import osmnx as ox
import geopandas as gpd
from shapely.geometry import Point

def get_distance_to_roads(lat, lon, radius_m=5000):
    try:
        # Query roads in EPSG:4326
        roads = ox.features_from_point((lat, lon), tags={'highway': True}, dist=radius_m)
        if roads.empty:
            return None

        # Convert to projected CRS (meters)
        roads = roads.to_crs(epsg=3857)
        fire_pt = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326").to_crs(3857).iloc[0]

        # Compute min distance (meters)
        return float(roads.distance(fire_pt).min())

    except Exception as e:
        print(f"❌ Road distance error at {lat},{lon}: {e}")
        return None


def get_distance_to_settlements(lat, lon, radius_m=5000):
    try:
        settlements = ox.features_from_point((lat, lon), tags={'building': True}, dist=radius_m)
        if settlements.empty:
            return None

        settlements = settlements.to_crs(epsg=3857)
        fire_pt = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326").to_crs(3857).iloc[0]

        return float(settlements.distance(fire_pt).min())

    except Exception as e:
        print(f"❌ Settlement distance error at {lat},{lon}: {e}")
        return None


In [12]:
import pandas as pd

records = []

for i, row in fire_gdf.iterrows():
    try:
        print(f"🔥 Processing fire {i+1}/{len(fire_gdf)} ...")

        lat, lon = row.geometry.y, row.geometry.x

        features = {
            'fire_id': i+1,
            'latitude': lat,
            'longitude': lon,
            'date': row['Dato anrop'],
            'fire_area': row['Total Area (decare)']
        }

        # --- Add features from each module ---
        features.update(extract_veg_temp_indices(row))   # NDVI, NDWI, MSI, Temperature
        features.update(extract_terrain_landcover(row))  # Elevation, slope, aspect, landcover, tree cover
        features.update(extract_weather(row))            # Humidity, windspeed, precip last 3 days
        features.update(extract_population(row))         # Population density

        # --- Distances from OSMnx ---
        features['DistanceRoad'] = get_distance_to_roads(lat, lon, radius_m=5000)
        features['DistanceSettlement'] = get_distance_to_settlements(lat, lon, radius_m=5000)

        records.append(features)

    except Exception as e:
        print(f"❌ Error processing fire {i+1}: {e}")

# --- Merge all records into DataFrame ---
df_all = pd.DataFrame(records)

# --- Reorder columns for consistency ---
column_order = [
    'fire_id', 'date', 'latitude', 'longitude', 'fire_area',
    'Elevation', 'Slope', 'Aspect', 'LandCover', 'TreeCover',
    'Population', 'DistanceRoad', 'DistanceSettlement',
    'Temperature', 'Humidity', 'WindSpeed', 'Precip3Days',
    'NDVI', 'NDWI', 'MSI'
]
df_all = df_all[[col for col in column_order if col in df_all.columns]]

# --- Save final file ---
output_file = "fire_parameters.xlsx"
df_all.to_excel(output_file, index=False)

print(f"✅ Fire features saved as {output_file}")
print("📊 Shape:", df_all.shape)
print("📑 Columns:", df_all.columns.tolist())


🔥 Processing fire 9486/9931 ...
🔥 Processing fire 9656/9931 ...
🔥 Processing fire 3244/9931 ...
🔥 Processing fire 9369/9931 ...
🔥 Processing fire 9591/9931 ...
🔥 Processing fire 7578/9931 ...
🔥 Processing fire 8008/9931 ...
🔥 Processing fire 2137/9931 ...
🔥 Processing fire 4581/9931 ...
🔥 Processing fire 3731/9931 ...
🔥 Processing fire 3757/9931 ...
🔥 Processing fire 2454/9931 ...
🔥 Processing fire 6765/9931 ...
🔥 Processing fire 6903/9931 ...
🔥 Processing fire 4218/9931 ...
❌ Error processing fire 4218: Expected a homogeneous image collection, but an image with incompatible bands was encountered:
  First image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, MSK_CLASSI_OPAQUE, MSK_CLASSI_CIRRUS, MSK_CLASSI_SNOW_ICE]).
Current image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, QA10, QA20, QA60]).
          Image ID: 20180627T104021_20180627T104023_T32VMM
Some bands might require explicit casts.
🔥 Processing fire 9886/9931 ...
🔥 Proces

🔥 Processing fire 9764/9931 ...
❌ Error processing fire 9764: No Sentinel-2 images available for this region and date.
🔥 Processing fire 2446/9931 ...
🔥 Processing fire 9716/9931 ...
❌ Error processing fire 9716: No Sentinel-2 images available for this region and date.
🔥 Processing fire 2388/9931 ...
🔥 Processing fire 1526/9931 ...
🔥 Processing fire 8546/9931 ...
🔥 Processing fire 129/9931 ...
🔥 Processing fire 1091/9931 ...
🔥 Processing fire 8743/9931 ...
🔥 Processing fire 2863/9931 ...
🔥 Processing fire 1391/9931 ...
❌ Error processing fire 1391: No Sentinel-2 images available for this region and date.
🔥 Processing fire 615/9931 ...
🔥 Processing fire 2618/9931 ...
🔥 Processing fire 2378/9931 ...
🔥 Processing fire 1044/9931 ...
❌ Error processing fire 1044: No Sentinel-2 images available for this region and date.
🔥 Processing fire 8798/9931 ...
🔥 Processing fire 6732/9931 ...
🔥 Processing fire 8126/9931 ...
🔥 Processing fire 681/9931 ...
❌ Error processing fire 681: No Sentinel-2 imag

🔥 Processing fire 6092/9931 ...
🔥 Processing fire 9618/9931 ...
🔥 Processing fire 6077/9931 ...
🔥 Processing fire 6267/9931 ...
🔥 Processing fire 6078/9931 ...
🔥 Processing fire 6115/9931 ...
❌ Error processing fire 6115: Expected a homogeneous image collection, but an image with incompatible bands was encountered:
  First image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, QA10, QA20, QA60]).
Current image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, MSK_CLASSI_OPAQUE, MSK_CLASSI_CIRRUS, MSK_CLASSI_SNOW_ICE]).
          Image ID: 20180625T105029_20180625T105253_T32VLL
Some bands might require explicit casts.
🔥 Processing fire 854/9931 ...
🔥 Processing fire 9400/9931 ...
🔥 Processing fire 8763/9931 ...
🔥 Processing fire 8912/9931 ...
🔥 Processing fire 3760/9931 ...
🔥 Processing fire 4266/9931 ...
❌ Error processing fire 4266: No Sentinel-2 images available for this region and date.
🔥 Processing fire 4263/9931 ...
🔥 Processing fire 

❌ Error processing fire 6337: Expected a homogeneous image collection, but an image with incompatible bands was encountered:
  First image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, QA10, QA20, QA60]).
Current image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, MSK_CLASSI_OPAQUE, MSK_CLASSI_CIRRUS, MSK_CLASSI_SNOW_ICE]).
          Image ID: 20180625T105029_20180625T105253_T32VNM
Some bands might require explicit casts.
🔥 Processing fire 6142/9931 ...
🔥 Processing fire 2300/9931 ...
🔥 Processing fire 8316/9931 ...
🔥 Processing fire 680/9931 ...
🔥 Processing fire 3593/9931 ...
🔥 Processing fire 6124/9931 ...
❌ Error processing fire 6124: Expected a homogeneous image collection, but an image with incompatible bands was encountered:
  First image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, QA10, QA20, QA60]).
Current image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, MSK_CLASSI

🔥 Processing fire 1602/9931 ...
❌ Error processing fire 1602: No Sentinel-2 images available for this region and date.
🔥 Processing fire 2955/9931 ...
🔥 Processing fire 7647/9931 ...
🔥 Processing fire 8671/9931 ...
❌ Error processing fire 8671: No Sentinel-2 images available for this region and date.
🔥 Processing fire 8838/9931 ...
🔥 Processing fire 9183/9931 ...
🔥 Processing fire 731/9931 ...
🔥 Processing fire 8844/9931 ...
🔥 Processing fire 563/9931 ...
🔥 Processing fire 3432/9931 ...
❌ Error processing fire 3432: Expected a homogeneous image collection, but an image with incompatible bands was encountered:
  First image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, QA10, QA20, QA60]).
Current image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, MSK_CLASSI_OPAQUE, MSK_CLASSI_CIRRUS, MSK_CLASSI_SNOW_ICE]).
          Image ID: 20180625T105029_20180625T105253_T32VNM
Some bands might require explicit casts.
🔥 Processing fire 8006/9931 

...

🔥 Processing fire 4213/9931 ...
🔥 Processing fire 4212/9931 ...
🔥 Processing fire 4211/9931 ...
🔥 Processing fire 4210/9931 ...
🔥 Processing fire 4209/9931 ...
🔥 Processing fire 4208/9931 ...
🔥 Processing fire 4205/9931 ...
🔥 Processing fire 4202/9931 ...
🔥 Processing fire 4201/9931 ...
🔥 Processing fire 4200/9931 ...
🔥 Processing fire 450/9931 ...
🔥 Processing fire 4244/9931 ...
❌ Error processing fire 4244: Expected a homogeneous image collection, but an image with incompatible bands was encountered:
  First image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, MSK_CLASSI_OPAQUE, MSK_CLASSI_CIRRUS, MSK_CLASSI_SNOW_ICE]).
Current image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, QA10, QA20, QA60]).
          Image ID: 20180625T105029_20180625T105253_T32VPM
Some bands might require explicit casts.
🔥 Processing fire 441/9931 ...
🔥 Processing fire 443/9931 ...
🔥 Processing fire 4278/9931 ...
❌ Error processing fire 4278: No Sentinel-2

❌ Error processing fire 3966: Expected a homogeneous image collection, but an image with incompatible bands was encountered:
  First image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, QA10, QA20, QA60]).
Current image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, MSK_CLASSI_OPAQUE, MSK_CLASSI_CIRRUS, MSK_CLASSI_SNOW_ICE]).
          Image ID: 20180625T105029_20180625T105253_T32VLL
Some bands might require explicit casts.
🔥 Processing fire 3963/9931 ...
🔥 Processing fire 3962/9931 ...
🔥 Processing fire 3960/9931 ...
🔥 Processing fire 3955/9931 ...
🔥 Processing fire 3953/9931 ...
🔥 Processing fire 3951/9931 ...
🔥 Processing fire 3950/9931 ...
❌ Error processing fire 3950: Expected a homogeneous image collection, but an image with incompatible bands was encountered:
  First image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B10, B11, B12, QA10, QA20, QA60]).
Current image type: 16 bands ([B1, B2, B3, B4, B5, B6, B7, B8, B

🔥 Processing fire 495/9931 ...
🔥 Processing fire 3863/9931 ...
🔥 Processing fire 3877/9931 ...
🔥 Processing fire 3876/9931 ...
🔥 Processing fire 3875/9931 ...
🔥 Processing fire 504/9931 ...
❌ Error processing fire 504: No Sentinel-2 images available for this region and date.
🔥 Processing fire 3873/9931 ...
🔥 Processing fire 506/9931 ...
🔥 Processing fire 3869/9931 ...
🔥 Processing fire 3867/9931 ...
🔥 Processing fire 3865/9931 ...
🔥 Processing fire 3864/9931 ...
🔥 Processing fire 3861/9931 ...
🔥 Processing fire 502/9931 ...
❌ Error processing fire 502: No Sentinel-2 images available for this region and date.
🔥 Processing fire 3860/9931 ...
🔥 Processing fire 3859/9931 ...
❌ Error processing fire 3859: No Sentinel-2 images available for this region and date.
🔥 Processing fire 3858/9931 ...
🔥 Processing fire 3857/9931 ...
🔥 Processing fire 3855/9931 ...
🔥 Processing fire 3854/9931 ...
🔥 Processing fire 3852/9931 ...
🔥 Processing fire 3851/9931 ...
🔥 Processing fire 3850/9931 ...
🔥 Process

🔥 Processing fire 4764/9931 ...
🔥 Processing fire 4893/9931 ...
🔥 Processing fire 4895/9931 ...
🔥 Processing fire 431/9931 ...
🔥 Processing fire 381/9931 ...
🔥 Processing fire 5035/9931 ...
🔥 Processing fire 5033/9931 ...
🔥 Processing fire 5031/9931 ...
🔥 Processing fire 5028/9931 ...
🔥 Processing fire 374/9931 ...
🔥 Processing fire 5024/9931 ...
🔥 Processing fire 5019/9931 ...
🔥 Processing fire 5018/9931 ...
🔥 Processing fire 5017/9931 ...
🔥 Processing fire 5014/9931 ...
🔥 Processing fire 5013/9931 ...
🔥 Processing fire 5011/9931 ...
🔥 Processing fire 5010/9931 ...
🔥 Processing fire 376/9931 ...
🔥 Processing fire 5008/9931 ...
🔥 Processing fire 5006/9931 ...
🔥 Processing fire 5000/9931 ...
🔥 Processing fire 377/9931 ...
🔥 Processing fire 4997/9931 ...
🔥 Processing fire 4996/9931 ...
🔥 Processing fire 4995/9931 ...
🔥 Processing fire 4994/9931 ...
🔥 Processing fire 4993/9931 ...
🔥 Processing fire 5036/9931 ...
🔥 Processing fire 5039/9931 ...
🔥 Processing fire 5040/9931 ...
🔥 Processing 

🔥 Processing fire 4440/9931 ...
❌ Error processing fire 4440: No Sentinel-2 images available for this region and date.
🔥 Processing fire 4439/9931 ...
❌ Error processing fire 4439: No Sentinel-2 images available for this region and date.
🔥 Processing fire 4438/9931 ...
❌ Error processing fire 4438: No Sentinel-2 images available for this region and date.
🔥 Processing fire 4437/9931 ...
❌ Error processing fire 4437: No Sentinel-2 images available for this region and date.
🔥 Processing fire 4436/9931 ...
❌ Error processing fire 4436: No Sentinel-2 images available for this region and date.
🔥 Processing fire 4433/9931 ...
🔥 Processing fire 4417/9931 ...
🔥 Processing fire 4430/9931 ...
🔥 Processing fire 4429/9931 ...
🔥 Processing fire 4427/9931 ...
🔥 Processing fire 4426/9931 ...
🔥 Processing fire 4424/9931 ...
🔥 Processing fire 4423/9931 ...
🔥 Processing fire 4422/9931 ...
🔥 Processing fire 4420/9931 ...
🔥 Processing fire 4419/9931 ...
❌ Error processing fire 4419: No Sentinel-2 images av

In [19]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# --- 1. Load non-fire Excel ---
nonfire_df = pd.read_excel("non_fire_events.xlsx")

# --- 2. Convert to GeoDataFrame ---
nonfire_gdf = gpd.GeoDataFrame(
    nonfire_df,
    geometry=gpd.points_from_xy(nonfire_df['longitude'], nonfire_df['latitude']),
    crs="EPSG:4326"
)

print("✅ Non-fire GeoDataFrame ready:", nonfire_gdf.shape)
print(nonfire_gdf.head())


✅ Non-fire GeoDataFrame ready: (8630, 7)
   nonfire_id   latitude  longitude       date  Cluster  fire_label  \
0           1  64.711433  13.323352 2021-07-17        0           0   
1           2  61.198462   6.776134 2022-08-09        0           0   
2           3  65.038281  13.153087 2016-10-05        0           0   
3           4  64.658855  12.292247 2018-04-02        0           0   
4           5  69.750828  22.973926 2021-04-13        0           0   

                    geometry  
0  POINT (13.32335 64.71143)  
1   POINT (6.77613 61.19846)  
2  POINT (13.15309 65.03828)  
3  POINT (12.29225 64.65885)  
4  POINT (22.97393 69.75083)  


In [20]:
for i, row in nonfire_gdf.iterrows():
    try:
        print(f"🌍 Processing non-fire {i+1}/{len(nonfire_gdf)} ...")

        lat, lon = row.geometry.y, row.geometry.x

        # Make a copy so functions still see "Dato anrop"
        row_firelike = row.copy()
        row_firelike['Dato anrop'] = row['date']   # ✅ map to expected column

        features = {
            'nonfire_id': i+1,
            'latitude': lat,
            'longitude': lon,
            'date': row['date'],
            'fire_area': 0,
            'fire_label': 0
        }

        # Use row_firelike so all functions work without edits
        features.update(extract_veg_temp_indices(row_firelike))
        features.update(extract_terrain_landcover(row_firelike))
        features.update(extract_weather(row_firelike))
        features.update(extract_population(row_firelike))

        features['DistanceRoad'] = get_distance_to_roads(lat, lon, radius_m=5000)
        features['DistanceSettlement'] = get_distance_to_settlements(lat, lon, radius_m=5000)

        records.append(features)

    except Exception as e:
        print(f"❌ Error processing non-fire {i+1}: {e}")


🌍 Processing non-fire 1/8630 ...
🌍 Processing non-fire 2/8630 ...
🌍 Processing non-fire 3/8630 ...
🌍 Processing non-fire 4/8630 ...
🌍 Processing non-fire 5/8630 ...
❌ Settlement distance error at 69.75082818616684,22.97392609870706: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 6/8630 ...
🌍 Processing non-fire 7/8630 ...
🌍 Processing non-fire 8/8630 ...
❌ Road distance error at 68.89513532584648,25.00113897193188: No matching features. Check query location, tags, and log.
❌ Settlement distance error at 68.89513532584648,25.00113897193188: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 9/8630 ...
❌ Error processing non-fire 9: No Sentinel-2 images available for this region and date.
🌍 Processing non-fire 10/8630 ...
❌ Error processing non-fire 10: No Sentinel-2 images available for this region and date.
🌍 Processing non-fire 11/8630 ...
🌍 Processing non-fire 12/8630 ...
❌ Error processing non-fire 12: No Sentinel-2 ima

❌ Error processing non-fire 106: No Sentinel-2 images available for this region and date.
🌍 Processing non-fire 107/8630 ...
🌍 Processing non-fire 108/8630 ...
🌍 Processing non-fire 109/8630 ...
🌍 Processing non-fire 110/8630 ...
🌍 Processing non-fire 111/8630 ...
🌍 Processing non-fire 112/8630 ...
🌍 Processing non-fire 113/8630 ...
🌍 Processing non-fire 114/8630 ...
🌍 Processing non-fire 115/8630 ...
❌ Road distance error at 70.5092436078263,25.40711217837307: No matching features. Check query location, tags, and log.
❌ Settlement distance error at 70.5092436078263,25.40711217837307: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 116/8630 ...
❌ Road distance error at 69.69277168510854,24.93467277341074: No matching features. Check query location, tags, and log.
❌ Settlement distance error at 69.69277168510854,24.93467277341074: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 117/8630 ...
🌍 Processing non-fire 118/8630 

🌍 Processing non-fire 219/8630 ...
🌍 Processing non-fire 220/8630 ...
🌍 Processing non-fire 221/8630 ...
❌ Error processing non-fire 221: No Sentinel-2 images available for this region and date.
🌍 Processing non-fire 222/8630 ...
🌍 Processing non-fire 223/8630 ...
🌍 Processing non-fire 224/8630 ...
🌍 Processing non-fire 225/8630 ...
🌍 Processing non-fire 226/8630 ...
🌍 Processing non-fire 227/8630 ...
🌍 Processing non-fire 228/8630 ...
🌍 Processing non-fire 229/8630 ...
🌍 Processing non-fire 230/8630 ...
🌍 Processing non-fire 231/8630 ...
🌍 Processing non-fire 232/8630 ...
🌍 Processing non-fire 233/8630 ...
❌ Road distance error at 70.36854525625886,23.94377519924715: No matching features. Check query location, tags, and log.
❌ Settlement distance error at 70.36854525625886,23.94377519924715: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 234/8630 ...
❌ Error processing non-fire 234: No Sentinel-2 images available for this region and date.
🌍 Processing

🌍 Processing non-fire 347/8630 ...
🌍 Processing non-fire 348/8630 ...
🌍 Processing non-fire 349/8630 ...
🌍 Processing non-fire 350/8630 ...
❌ Road distance error at 70.57492359840614,25.59613557541683: No matching features. Check query location, tags, and log.
❌ Settlement distance error at 70.57492359840614,25.59613557541683: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 351/8630 ...
🌍 Processing non-fire 352/8630 ...
🌍 Processing non-fire 353/8630 ...
🌍 Processing non-fire 354/8630 ...
❌ Road distance error at 68.48607158410782,19.45527563095286: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 355/8630 ...
❌ Error processing non-fire 355: No Sentinel-2 images available for this region and date.
🌍 Processing non-fire 356/8630 ...
🌍 Processing non-fire 357/8630 ...
🌍 Processing non-fire 358/8630 ...
❌ Road distance error at 70.53885763412278,26.81024570606108: No matching features. Check query location, tags, and log.


🌍 Processing non-fire 466/8630 ...
❌ Road distance error at 70.58096096539312,28.12894996929672: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 467/8630 ...
🌍 Processing non-fire 468/8630 ...
🌍 Processing non-fire 469/8630 ...
🌍 Processing non-fire 470/8630 ...
🌍 Processing non-fire 471/8630 ...
🌍 Processing non-fire 472/8630 ...
🌍 Processing non-fire 473/8630 ...
🌍 Processing non-fire 474/8630 ...
🌍 Processing non-fire 475/8630 ...
🌍 Processing non-fire 476/8630 ...
🌍 Processing non-fire 477/8630 ...
🌍 Processing non-fire 478/8630 ...
🌍 Processing non-fire 479/8630 ...
🌍 Processing non-fire 480/8630 ...
🌍 Processing non-fire 481/8630 ...
🌍 Processing non-fire 482/8630 ...
❌ Settlement distance error at 69.3649323739081,22.21953317636266: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 483/8630 ...
🌍 Processing non-fire 484/8630 ...
🌍 Processing non-fire 485/8630 ...
❌ Road distance error at 67.22108109426058,13.9890862

...

🌍 Processing non-fire 8058/8630 ...
❌ Error processing non-fire 8058: No Sentinel-2 images available for this region and date.
🌍 Processing non-fire 8059/8630 ...
❌ Error processing non-fire 8059: No Sentinel-2 images available for this region and date.
🌍 Processing non-fire 8060/8630 ...
🌍 Processing non-fire 8061/8630 ...
🌍 Processing non-fire 8062/8630 ...
🌍 Processing non-fire 8063/8630 ...
❌ Road distance error at 69.70120991594592,21.73759617293941: No matching features. Check query location, tags, and log.
❌ Settlement distance error at 69.70120991594592,21.73759617293941: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 8064/8630 ...
🌍 Processing non-fire 8065/8630 ...
🌍 Processing non-fire 8066/8630 ...
🌍 Processing non-fire 8067/8630 ...
🌍 Processing non-fire 8068/8630 ...
🌍 Processing non-fire 8069/8630 ...
🌍 Processing non-fire 8070/8630 ...
🌍 Processing non-fire 8071/8630 ...
🌍 Processing non-fire 8072/8630 ...
🌍 Processing non-fire 8073/863

🌍 Processing non-fire 8169/8630 ...
❌ Error processing non-fire 8169: No Sentinel-2 images available for this region and date.
🌍 Processing non-fire 8170/8630 ...
🌍 Processing non-fire 8171/8630 ...
❌ Error processing non-fire 8171: No Sentinel-2 images available for this region and date.
🌍 Processing non-fire 8172/8630 ...
❌ Road distance error at 70.13089638930197,26.3205844685929: No matching features. Check query location, tags, and log.
❌ Settlement distance error at 70.13089638930197,26.3205844685929: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 8173/8630 ...
❌ Settlement distance error at 69.78190987749934,24.37494590721646: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 8174/8630 ...
❌ Error processing non-fire 8174: No Sentinel-2 images available for this region and date.
🌍 Processing non-fire 8175/8630 ...
🌍 Processing non-fire 8176/8630 ...
🌍 Processing non-fire 8177/8630 ...
🌍 Processing non-fire 8178/863

🌍 Processing non-fire 8271/8630 ...
🌍 Processing non-fire 8272/8630 ...
🌍 Processing non-fire 8273/8630 ...
❌ Error processing non-fire 8273: No Sentinel-2 images available for this region and date.
🌍 Processing non-fire 8274/8630 ...
❌ Error processing non-fire 8274: No Sentinel-2 images available for this region and date.
🌍 Processing non-fire 8275/8630 ...
❌ Road distance error at 70.22681160286842,26.36782054238892: No matching features. Check query location, tags, and log.
❌ Settlement distance error at 70.22681160286842,26.36782054238892: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 8276/8630 ...
🌍 Processing non-fire 8277/8630 ...
🌍 Processing non-fire 8278/8630 ...
🌍 Processing non-fire 8279/8630 ...
🌍 Processing non-fire 8280/8630 ...
🌍 Processing non-fire 8281/8630 ...
🌍 Processing non-fire 8282/8630 ...
🌍 Processing non-fire 8283/8630 ...
🌍 Processing non-fire 8284/8630 ...
❌ Error processing non-fire 8284: No Sentinel-2 images available f

🌍 Processing non-fire 8409/8630 ...
❌ Road distance error at 68.8439827852115,24.91330727339287: No matching features. Check query location, tags, and log.
❌ Settlement distance error at 68.8439827852115,24.91330727339287: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 8410/8630 ...
🌍 Processing non-fire 8411/8630 ...
❌ Error processing non-fire 8411: No Sentinel-2 images available for this region and date.
🌍 Processing non-fire 8412/8630 ...
🌍 Processing non-fire 8413/8630 ...
🌍 Processing non-fire 8414/8630 ...
🌍 Processing non-fire 8415/8630 ...
🌍 Processing non-fire 8416/8630 ...
🌍 Processing non-fire 8417/8630 ...
🌍 Processing non-fire 8418/8630 ...
🌍 Processing non-fire 8419/8630 ...
🌍 Processing non-fire 8420/8630 ...
❌ Road distance error at 70.44389896005165,30.08795759440468: No matching features. Check query location, tags, and log.
❌ Settlement distance error at 70.44389896005165,30.08795759440468: No matching features. Check query location

🌍 Processing non-fire 8535/8630 ...
🌍 Processing non-fire 8536/8630 ...
🌍 Processing non-fire 8537/8630 ...
🌍 Processing non-fire 8538/8630 ...
❌ Error processing non-fire 8538: No Sentinel-2 images available for this region and date.
🌍 Processing non-fire 8539/8630 ...
🌍 Processing non-fire 8540/8630 ...
🌍 Processing non-fire 8541/8630 ...
❌ Road distance error at 69.97681316799206,28.64633789297639: No matching features. Check query location, tags, and log.
❌ Settlement distance error at 69.97681316799206,28.64633789297639: No matching features. Check query location, tags, and log.
🌍 Processing non-fire 8542/8630 ...
🌍 Processing non-fire 8543/8630 ...
🌍 Processing non-fire 8544/8630 ...
🌍 Processing non-fire 8545/8630 ...
🌍 Processing non-fire 8546/8630 ...
🌍 Processing non-fire 8547/8630 ...
🌍 Processing non-fire 8548/8630 ...
🌍 Processing non-fire 8549/8630 ...
🌍 Processing non-fire 8550/8630 ...
🌍 Processing non-fire 8551/8630 ...
🌍 Processing non-fire 8552/8630 ...
🌍 Processing 

In [21]:
# --- After the loop ---
import pandas as pd

# Merge all non-fire records into DataFrame
nonfire_df = pd.DataFrame(records)

# Reorder columns for consistency with fire file
column_order = [
    'nonfire_id', 'date', 'latitude', 'longitude', 'fire_area', 'fire_label',
    'Elevation', 'Slope', 'Aspect', 'LandCover', 'TreeCover',
    'Population', 'DistanceRoad', 'DistanceSettlement',
    'Temperature', 'Humidity', 'WindSpeed', 'Precip3Days',
    'NDVI', 'NDWI', 'MSI'
]
nonfire_df = nonfire_df[[col for col in column_order if col in nonfire_df.columns]]

# Save to Excel
output_file = "non_fire_parameters.xlsx"
nonfire_df.to_excel(output_file, index=False)

print(f"✅ Non-fire features saved as {output_file}")
print("📊 Shape:", nonfire_df.shape)
print("📑 Columns:", nonfire_df.columns.tolist())


✅ Non-fire features saved as non_fire_parameters.xlsx
📊 Shape: (23616, 21)
📑 Columns: ['nonfire_id', 'date', 'latitude', 'longitude', 'fire_area', 'fire_label', 'Elevation', 'Slope', 'Aspect', 'LandCover', 'TreeCover', 'Population', 'DistanceRoad', 'DistanceSettlement', 'Temperature', 'Humidity', 'WindSpeed', 'Precip3Days', 'NDVI', 'NDWI', 'MSI']
